In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
universe = [
    "NVDA",  # NVIDIA
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "GOOGL", # Alphabet
    "AMZN",  # Amazon
    "META"   # Meta
]

In [3]:
universe = ['AAPL', 'JPM', 'JNJ', 'XOM', 'WMT', 'CAT']

In [4]:
def log_returns_data(universe):

  data=pd.DataFrame()

  for i in universe:
    ticker=yf.Ticker(i)
    df=ticker.history(period='10y')
    name=f"log_returns_{i}"
    data[name]=np.log(df['Close']/df['Close'].shift(1))

  return data.dropna()

In [5]:
data=log_returns_data(universe)
data.tail()

,log_returns_AAPL,log_returns_JPM,log_returns_JNJ,log_returns_XOM,log_returns_WMT,log_returns_CAT
Date,,,,,,
2026-08-31 00:00:00-04:00,-0.008955,-0.004484,-0.008204,0.026697,0.017119,-0.003480
2026-09-01 00:00:00-04:00,0.025797,-0.003010,0.019887,0.022121,0.009963,-0.023228
2026-09-02 00:00:00-04:00,-0.000523,0.003572,0.014715,-0.002434,0.001604,0.016699
2026-09-03 00:00:00-04:00,0.009952,0.016261,0.011632,-0.011889,0.021725,0.009872
2026-09-04 00:00:00-04:00,-0.025426,-0.009491,-0.011560,-0.017036,-0.011876,0.017100


In [6]:
stock_cols=data.columns

In [7]:
def compute_rolling_eigenvalues(data, window=60):
    leading_eigenvalues = []
    second_eigenvalues = []
    third_eigenvalues = []   # <-- new
    dates = []

    for i in range(window, len(data)):
        window_data = data.iloc[i-window:i]
        corr_matrix = window_data.corr()

        eigenvalues = np.linalg.eigvalsh(corr_matrix)
        eigenvalues_sorted = eigenvalues[::-1]

        leading_eigenvalues.append(eigenvalues_sorted[0])
        second_eigenvalues.append(eigenvalues_sorted[1])
        third_eigenvalues.append(eigenvalues_sorted[2])   # <-- new
        dates.append(data.index[i])

    eigen_df = pd.DataFrame({
        'leading_eigenvalue': leading_eigenvalues,
        'second_eigenvalue': second_eigenvalues,
        'third_eigenvalue': third_eigenvalues   # <-- new
    }, index=dates)

    eigen_df['eigenvalue_ratio'] = eigen_df['leading_eigenvalue'] / eigen_df['second_eigenvalue']
    eigen_df.index = eigen_df.index.tz_localize(None)


    return eigen_df


eigen_df=compute_rolling_eigenvalues(data,60)

In [8]:
def add_vol_desperssion_vix(n):
  n['market_realized_vol_20d']=n.mean(axis=1).rolling(20).std()*np.sqrt(252)
  n['market_cross_sectional_dispersion']=n.std(axis=1)
  n['VIX']=n.mean(axis=1).rolling(20).std()

  return n

ndata=add_vol_desperssion_vix(data.copy())

In [9]:
def final(e,n):
  final_data=n.merge(e,how='inner',right_index=True,left_index=True)
  return final_data

# Ensure ndata's index is timezone-naive before merging
ndata.index = ndata.index.tz_localize(None)
final_data=final(eigen_df.copy(),ndata.copy())

In [10]:
final_data['vix_-1']=final_data['VIX'].shift(-1)
final_data.dropna(inplace=True)

In [11]:
def vix_model(data):
  data.drop((data.select_dtypes(include='object').columns),axis=1,inplace=True)

  from sklearn.model_selection import train_test_split as tts , GridSearchCV as grid ,RandomizedSearchCV as rand
  from sklearn.linear_model import LinearRegression
  from sklearn.ensemble import RandomForestRegressor,VotingRegressor
  from xgboost import XGBRegressor
  from sklearn.neighbors import KNeighborsRegressor
  from sklearn.naive_bayes import GaussianNB
  from sklearn.metrics import accuracy_score,classification_report,r2_score
  from sklearn.tree import DecisionTreeRegressor
  from lightgbm import LGBMRegressor
  train=data[:1800]
  test=data[1800:]

  x=train.drop(['VIX','vix_-1'],axis=1)
  y=train['vix_-1']

  xtrain,xtest,ytrain,ytest=tts(x,y,shuffle=False,test_size=0.2)

  model=VotingRegressor([
    ('rfr',RandomForestRegressor(random_state=42)),
    ('xgb',XGBRegressor(random_state=42)),
    ('lr',LinearRegression()),
  ])
  model.fit(xtrain,ytrain)

  pred=model.predict(data.drop(['VIX','vix_-1'],axis=1))

  return pred

col=vix_model(final_data)
col

array([0.00655658, 0.00643164, 0.00639663, ..., 0.00670192, 0.00632305,
       0.00644232])

In [12]:
final_data['pred_vix']=col

In [13]:
lower=final_data['pred_vix'].quantile(0.33)
upper=final_data['pred_vix'].quantile(0.66)

l=[]
for i in final_data['pred_vix']:
  if i<=lower:
    l.append('calm')
  elif lower<i<=upper:
    l.append('transitioning')
  else:
    l.append('stressed')
final_data['regime']=l

import numpy as np
import pandas as pd

# --- Third eigenvalue (requires modifying compute_rolling_eigenvalues to also store it) ---
# If not already done, add this inside that function's loop:
#     third_eigenvalues.append(eigenvalues_sorted[2])
# and this to eigen_df construction:
#     eigen_df['third_eigenvalue'] = third_eigenvalues
# Then it flows into final_data automatically via the merge you already did.


# --- Rolling slope helper ---
def rolling_slope(series, window=10):
    return series.rolling(window).apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True
    )
# Rate-of-change features — capture the dynamics of regime shift, not just the level
final_data['eigenvalue_ratio_diff5'] = final_data['eigenvalue_ratio'].diff(5)
final_data['vol_diff5'] = final_data['market_realized_vol_20d'].diff(5)
final_data['dispersion_diff5'] = final_data['market_cross_sectional_dispersion'].diff(5)

# diff() introduces NaN for the first 5 rows (no prior value to compare agains
# --- New features ---
final_data['eigenvalue_ratio_slope10'] = rolling_slope(final_data['eigenvalue_ratio'], 10)
final_data['vol_slope10'] = rolling_slope(final_data['market_realized_vol_20d'], 10)
final_data['eigenvalue_gap_2_3'] = final_data['second_eigenvalue'] - final_data['third_eigenvalue']
final_data['vol_of_vol_10d'] = final_data['market_realized_vol_20d'].rolling(10).std()
final_data['market_return'] = final_data.select_dtypes(include=np.number).mean(axis=1)
final_data['market_return_skew_20d'] = final_data['market_return'].rolling(20).skew()
# 3. Rolling correlation between volatility and dispersion — divergence can signal transition
final_data['vol_dispersion_corr_20d'] = final_data['market_realized_vol_20d'].rolling(20).corr(
    final_data['market_cross_sectional_dispersion']
)

# 4. Return autocorrelation — crashes/panics often show negative autocorrelation (sharp reversals)
final_data['return_autocorr_20d'] = final_data['market_return'].rolling(20).apply(
    lambda x: pd.Series(x).autocorr(lag=1), raw=False
)

# 5. Rolling z-score of today's volatility vs its own recent history — "how unusual is today specifically"
final_data['vol_zscore_60d'] = (
    final_data['market_realized_vol_20d'] - final_data['market_realized_vol_20d'].rolling(60).mean()
) / final_data['market_realized_vol_20d'].rolling(60).std()

# 6. Dispersion z-score, same idea, for cross-sectional dispersion
final_data['dispersion_zscore_60d'] = (
    final_data['market_cross_sectional_dispersion'] - final_data['market_cross_sectional_dispersion'].rolling(60).mean()
) / final_data['market_cross_sectional_dispersion'].rolling(60).std()

cum_return = final_data['market_return'].cumsum()
rolling_max = cum_return.rolling(20).max()
final_data['drawdown_20d'] = cum_return - rolling_max

downside_returns = final_data['market_return'].clip(upper=0)
final_data['downside_vol_20d'] = downside_returns.rolling(20).std()

final_data = final_data.dropna()

In [14]:
feature_cols = ['market_realized_vol_20d', 'market_cross_sectional_dispersion',
                 'downside_vol_20d', 'leading_eigenvalue', 'eigenvalue_ratio',
                 'vol_of_vol_10d', 'eigenvalue_ratio_slope10', 'market_return_skew_20d',
                 'vol_zscore_60d', 'dispersion_zscore_60d',
                 'eigenvalue_ratio_diff5', 'vol_diff5', 'dispersion_diff5',
                 'vol_dispersion_corr_20d', 'vol_slope10']

In [15]:
feature_cols = ['market_realized_vol_20d', 'market_cross_sectional_dispersion',
                 'leading_eigenvalue', 'second_eigenvalue', 'eigenvalue_ratio',
                 'eigenvalue_ratio_diff5', 'vol_diff5', 'dispersion_diff5']

# **CLASSIFICATION MODEL**

In [16]:
from sklearn.model_selection import train_test_split as tts , GridSearchCV as grid ,RandomizedSearchCV as rand
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,classification_report
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier

final features

In [17]:
train=final_data[:1800]
test=final_data[1800:]

In [18]:
x=train[feature_cols]
y=train['regime']

xtrain,xtest,ytrain,ytest=tts(x,y,shuffle=False,test_size=0.2)

In [19]:
model = VotingClassifier([
    ('rfc', RandomForestClassifier(random_state=42)),
    ('xgb', XGBClassifier(random_state=42)),
    ('lgb',LGBMClassifier(random_state=42))
], voting='hard')

In [20]:
model.fit(xtrain,ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 1440, number of used features: 8
[LightGBM] [Info] Start training from score -1.278437
[LightGBM] [Info] Start training from score -0.774116
[LightGBM] [Info] Start training from score -1.345472
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


VotingClassifier(estimators=[('rfc', RandomForestClassifier(random_state=42)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=True,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_polic...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                             ('lgb', LGBMClassifier(random_state=42))])

In [21]:
model.score(xtest,ytest)

0.8694444444444445

In [22]:
pred=model.predict(xtest)
round(accuracy_score(ytest,pred),2)

0.87

In [23]:
print(classification_report(ytest, model.predict(xtest)))

               precision    recall  f1-score   support

         calm       0.87      0.86      0.86       153
     stressed       0.95      0.98      0.96        81
transitioning       0.81      0.82      0.81       126

     accuracy                           0.87       360
    macro avg       0.88      0.88      0.88       360
 weighted avg       0.87      0.87      0.87       360



In [24]:
x=test[feature_cols]
y=test['regime']

pred=model.predict(x)
print((accuracy_score(y,pred)))

0.8718381112984823


In [25]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
import numpy as np

X = final_data[feature_cols]
y = final_data['regime']

tscv = TimeSeriesSplit(n_splits=5)

scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    scores.append(acc)

    print(f"Fold {fold+1}: accuracy = {acc:.3f}, train size = {len(X_train)}, test size = {len(X_test)}")

print()
print(f"Mean accuracy: {np.mean(scores):.3f}")
print(f"Std deviation: {np.std(scores):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1080
[LightGBM] [Info] Number of data points in the train set: 403, number of used features: 8
[LightGBM] [Info] Start training from score -0.419207
[LightGBM] [Info] Start training from score -1.694871
[LightGBM] [Info] Start training from score -1.840053
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

# **PORTFOLIO OVERLAY**

**LAYER - 1**

**The covariance matrix**

This is the mathematical core of portfolio risk. Here's why you need it,
conceptually: a portfolio's total risk isn't just "add up each stock's individual risk" — it depends on how stocks move together. If two stocks always move in opposite directions, holding both actually reduces your combined risk (they cancel out); if they always move together, holding both doesn't reduce risk much at all. Covariance is the mathematical object that captures both each stock's own variance (on the diagonal) and every pair's co-movement (off-diagonal) — it's the input every portfolio optimization formula needs. .cov() computes this automatically from your historical returns; .values just converts it from a pandas DataFrame into a plain numpy array, which is the format CVXPY expects.

**Setting up the optimization variables**

cp.Variable(n) tells CVXPY: "there are 6 unknown numbers I want you to solve for." These 6 numbers (w) represent portfolio weights — what fraction of your total money sits in each stock. Right now they're undefined; the solver's job is to find the actual values.


**The objective — what "minimum variance" actually means mathematically**

This is the standard formula for portfolio variance: wᵀ Σ w (w-transpose times the covariance matrix times w). Intuitively: it takes your proposed weights, multiplies them through the covariance matrix, and produces a single number representing how much total risk that specific weight combination produces. cp.quad_form is just CVXPY's built-in shortcut for this exact formula, so you don't have to write out matrix multiplication by hand.

**The constraints — the "rules" the solution must obey**

`cp.sum(w) == 1` — your weights must add up to 100% of your capital (you're not leaving money uninvested, nor using leverage beyond your capital).
`w >= 0` — no negative weights, which means no short-selling — you can only buy stocks, not bet against them. This matches your project's design ("long-only min-variance," per the original plan).

**Solving it**

This tells CVXPY: "find the values of w that make portfolio_variance as small as possible, while still satisfying both constraints above." CVXPY runs a numerical optimization algorithm internally (you don't need to know its mechanics) and stores the answer.

**LAYER - 2**

Concept first: np.linalg.eigh (not eigvalsh) returns both eigenvalues AND their corresponding eigenvectors together. The leading eigenvector is a 6-dimensional direction — think of it as "if the whole market moves as one block, this vector describes each stock's relative contribution to that block movement." A portfolio that's very "aligned" with this direction is one that's heavily exposed to broad market co-movement; a portfolio that's more "orthogonal" to it is more diversified against that specific risk.
Code — extracting today's leading eigenvector (single snapshot, not the full rolling loop yet)



What this does: eigenvectors comes back as a matrix where each column is one eigenvector. Since eigh sorts eigenvalues smallest-to-largest (same convention as eigvalsh before), the very last column corresponds to the largest eigenvalue — the leading eigenvector. This 6-number vector is what we'll use to constrain the portfolio.
Run this and share the 6 values that come out — then I'll explain exactly how we turn this into a portfolio constraint (projecting the portfolio weights onto this direction, and capping that projection when the regime is transitioning/stressed).

In [26]:
import numpy as np
import cvxpy as cp
import pandas as pd

stock_names = universe

n = len(stock_cols)

# --- 1. Covariance matrix (for portfolio variance) ---
cov_matrix = final_data[stock_cols].cov().values

# --- 2. Correlation matrix + leading eigenvector (for the regime-aware constraint) ---
recent_window = final_data[stock_cols].iloc[-60:]
corr_matrix = recent_window.corr()

eigenvalues, eigenvectors = np.linalg.eigh(corr_matrix)
leading_eigenvector = eigenvectors[:, -1]   # eigh returns ascending order; last column = leading
v = leading_eigenvector

# --- 3. Baseline portfolio (no regime awareness) ---
w = cp.Variable(n)
portfolio_variance = cp.quad_form(w, cov_matrix)
constraints = [cp.sum(w) == 1, w >= 0]
problem = cp.Problem(cp.Minimize(portfolio_variance), constraints)
problem.solve()

baseline = w.value

# --- 4. Constrained portfolio (regime = stressed, cap exposure to leading eigenvector) ---
w3 = cp.Variable(n)
portfolio_variance3 = cp.quad_form(w3, cov_matrix)
exposure3 = w3 @ v

constraints3 = [
    cp.sum(w3) == 1,
    w3 >= 0,
    cp.abs(exposure3) <= 0.15   # tunable threshold
]
problem3 = cp.Problem(cp.Minimize(portfolio_variance3), constraints3)
problem3.solve()

constrained = w3.value

# --- 5. Comparison table ---
def classify_change(base, new):
    diff = new - base
    pct_of_base = diff / base if base != 0 else 0

    if abs(pct_of_base) < 0.05:
        return "~flat"
    elif pct_of_base >= 0.5:
        return "↑↑ (big jump)"
    elif pct_of_base > 0:
        return "↑"
    elif pct_of_base <= -0.5:
        return "↓↓ (big drop)"
    else:
        return "↓"

comparison = pd.DataFrame({
    'Stock': stock_names,
    'Baseline': np.round(baseline, 3),
    'Constrained': np.round(constrained, 3),
})
comparison['Change'] = [classify_change(b, c) for b, c in zip(baseline, constrained)]

print("Exposure (baseline):   ", baseline @ v)
print("Exposure (constrained):", constrained @ v)
print("Variance (baseline):   ", problem.value)
print("Variance (constrained):", problem3.value)
print()
print(comparison.to_string(index=False))

Exposure (baseline):    -0.39814183025929545
Exposure (constrained): -0.14999999999999997
Variance (baseline):    9.800713988520978e-05
Variance (constrained): 0.000116212662119134

Stock  Baseline  Constrained        Change
 AAPL     0.064        0.026 ↓↓ (big drop)
  JPM     0.053        0.159 ↑↑ (big jump)
  JNJ     0.447        0.330             ↓
  XOM     0.113        0.010 ↓↓ (big drop)
  WMT     0.287        0.243             ↓
  CAT     0.036        0.233 ↑↑ (big jump)


In [27]:
import yfinance as yf
import numpy as np
import pandas as pd

def latest_feature_extraction(tickers, window=60, vol_window=20):
    stock_cols = [f'log_returns_{t}' for t in tickers]

    # 1. Pull enough history to fill all rolling windows (60-day eigenvalue window + buffer)
    prices = yf.download(tickers, period='2y')['Close'][:-2]
    prices.columns = [f'log_returns_{t}' for t in prices.columns]

    # 2. Log returns
    data = np.log(prices / prices.shift(1)).dropna()
    data['VIX']=data.mean(axis=1).rolling(20).std()
    # 3. Market return, vol, dispersion
    data['market_return'] = data[stock_cols].mean(axis=1)
    data['market_realized_vol_20d'] = data['market_return'].rolling(vol_window).std() * np.sqrt(252)
    data['market_cross_sectional_dispersion'] = data[stock_cols].std(axis=1)

    # 4. Rolling eigenvalues (leading, second, third) + ratio
    leading_eig, second_eig, third_eig, dates = [], [], [], []
    for i in range(window, len(data)):
        window_data = data[stock_cols].iloc[i-window:i]
        corr_matrix = window_data.corr()
        eigenvalues = np.linalg.eigvalsh(corr_matrix)
        eigenvalues_sorted = eigenvalues[::-1]
        leading_eig.append(eigenvalues_sorted[0])
        second_eig.append(eigenvalues_sorted[1])
        third_eig.append(eigenvalues_sorted[2])
        dates.append(data.index[i])

    eigen_df = pd.DataFrame({
        'leading_eigenvalue': leading_eig,
        'second_eigenvalue': second_eig,
        'third_eigenvalue': third_eig
    }, index=dates)
    eigen_df['eigenvalue_ratio'] = eigen_df['leading_eigenvalue'] / eigen_df['second_eigenvalue']

    data = data.merge(eigen_df, left_index=True, right_index=True, how='inner')

    # 5. Diff / slope / gap / vol-of-vol / skew / dispersion-corr / autocorr / z-scores / drawdown / downside vol
    def rolling_slope(series, w=10):
        return series.rolling(w).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True)

    data['eigenvalue_ratio_diff5'] = data['eigenvalue_ratio'].diff(5)
    data['vol_diff5'] = data['market_realized_vol_20d'].diff(5)
    data['dispersion_diff5'] = data['market_cross_sectional_dispersion'].diff(5)
    data['eigenvalue_ratio_slope10'] = rolling_slope(data['eigenvalue_ratio'], 10)
    data['vol_slope10'] = rolling_slope(data['market_realized_vol_20d'], 10)
    data['eigenvalue_gap_2_3'] = data['second_eigenvalue'] - data['third_eigenvalue']
    data['vol_of_vol_10d'] = data['market_realized_vol_20d'].rolling(10).std()
    data['market_return_skew_20d'] = data['market_return'].rolling(20).skew()
    data['vol_dispersion_corr_20d'] = data['market_realized_vol_20d'].rolling(20).corr(data['market_cross_sectional_dispersion'])
    data['return_autocorr_20d'] = data['market_return'].rolling(20).apply(lambda x: pd.Series(x).autocorr(lag=1), raw=False)
    data['vol_zscore_60d'] = (data['market_realized_vol_20d'] - data['market_realized_vol_20d'].rolling(60).mean()) / data['market_realized_vol_20d'].rolling(60).std()
    data['dispersion_zscore_60d'] = (data['market_cross_sectional_dispersion'] - data['market_cross_sectional_dispersion'].rolling(60).mean()) / data['market_cross_sectional_dispersion'].rolling(60).std()

    cum_return = data['market_return'].cumsum()
    data['drawdown_20d'] = cum_return - cum_return.rolling(20).max()
    data['downside_vol_20d'] = data['market_return'].clip(upper=0).rolling(20).std()

    data = data.dropna()

    # 6. Return only the latest available row
    return data.iloc[[-1]]

In [28]:
tickers = universe
latest_row = latest_feature_extraction(tickers)
print(latest_row)

/tmp/ipykernel_1983/3565067982.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, period='2y')['Close'][:-2]
[*********************100%***********************]  6 of 6 completed


            log_returns_AAPL  log_returns_CAT  log_returns_JNJ  \
2026-09-02         -0.000523         0.016699         0.014715   

            log_returns_JPM  log_returns_WMT  log_returns_XOM       VIX  \
2026-09-02         0.003572         0.001604        -0.002434  0.007424   

            market_return  market_realized_vol_20d  \
2026-09-02       0.005605                 0.117858   

            market_cross_sectional_dispersion  ...  vol_slope10  \
2026-09-02                           0.008104  ...    -0.002786   

            eigenvalue_gap_2_3  vol_of_vol_10d  market_return_skew_20d  \
2026-09-02            0.064819        0.008783               -1.967568   

            vol_dispersion_corr_20d  return_autocorr_20d  vol_zscore_60d  \
2026-09-02                 0.063285             0.119784       -0.104968   

            dispersion_zscore_60d  drawdown_20d  downside_vol_20d  
2026-09-02              -1.138828     -0.016774          0.005607  

[1 rows x 28 columns]


In [29]:
latest_features=latest_row[feature_cols]

In [30]:
# Get the most recent row's features (or any specific date you want to check)
#latest_features = final_data[feature_cols].iloc[[-1]]   # double brackets keep it as a DataFrame


regime_tomorrow = model.predict(latest_features)[0]
print("Predicted regime for TOMORROW:", regime_tomorrow)

# Same constraint logic as before, now driven by the real prediction
if regime_tomorrow in ['transitioning', 'stressed']:
    constraints3 = [
        cp.sum(w3) == 1,
        w3 >= 0,
        cp.abs(exposure3) <= 0.15
    ]
else:
    constraints3 = [
        cp.sum(w3) == 1,
        w3 >= 0
        # no exposure cap when regime is calm
    ]

problem3 = cp.Problem(cp.Minimize(portfolio_variance3), constraints3)
problem3.solve()

Predicted regime for TOMORROW: transitioning


np.float64(0.000116212662119134)

In [31]:
problem3.solve()
print("Portfolio weights:", w3.value)
print("Portfolio variance:", problem3.value)

Portfolio weights: [0.02590968 0.15896371 0.32977993 0.00950186 0.24275456 0.23309025]
Portfolio variance: 0.00011621266211913426


In [32]:
latest_features

,market_realized_vol_20d,market_cross_sectional_dispersion,leading_eigenvalue,second_eigenvalue,eigenvalue_ratio,eigenvalue_ratio_diff5,vol_diff5,dispersion_diff5
2026-09-02,0.117858,0.008104,1.908588,1.19076,1.602832,0.152501,-0.010731,-0.004062


In [33]:
s=[0.02691703, 0.15962715, 0.32908552 ,0.00976842 ,0.24215025 ,0.23245162]

In [34]:
from sklearn.metrics import r2_score

In [45]:
print(f'score between the nowcast prediction for september 3rd and forecast model prediction \n for september 3rd on the basis of 2nd september values{r2_score(s,w3.value)}')

score between the nowcast prediction for september 3rd and forecast model prediction 
 for september 3rd on the basis of 2nd september values0.9999654945268012
